# Solution A2 · Confusion matrix, ROC curve and average precision from scratch

**Type**: Implement from scratch

## Objective

Implement with numpy the metrics of lesson 4.3: confusion matrix, precision/recall/F1, ROC curve and AUC (in two ways), average precision and the
search for the threshold that maximizes F1, correctly handling **tied scores**. Compare everything with scikit-learn.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
df["ratio"] = df["amount"] / df["income"]
cols = ["ratio", "age", "job_tenure", "income"]
X_tr, X_te, y_tr, y_te = train_test_split(df[cols], df["defaulted"], test_size=0.3, stratify=df["defaulted"], random_state=0)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
y_te = y_te.to_numpy()
scores = np.round(model.predict_proba(X_te)[:, 1], 2)
print(len(scores), "scores,", len(np.unique(scores)), "distinct values")

## Commented solution

In [ ]:
def confusion_matrix_(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    tn = int(((y == 0) & (pred == 0)).sum()); fp = int(((y == 0) & (pred == 1)).sum())
    fn = int(((y == 1) & (pred == 0)).sum()); tp = int(((y == 1) & (pred == 1)).sum())
    return tn, fp, fn, tp


def metrics(y, pred):
    tn, fp, fn, tp = confusion_matrix_(y, pred)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"accuracy": (tp + tn) / len(y), "precision": precision, "recall": recall, "f1": f1}

In [ ]:
def _cumulate(y, s):
    """Cumulative true and false positives for decreasing threshold, one value per distinct score."""
    y, s = np.asarray(y), np.asarray(s)
    order = np.argsort(-s, kind="stable")
    s_o, y_o = s[order], y[order]
    last = np.r_[np.where(np.diff(s_o) != 0)[0], len(s_o) - 1]        # one point for every group of ties
    return np.cumsum(y_o)[last], np.cumsum(1 - y_o)[last]


def roc_curve_(y, s):
    tps, fps = _cumulate(y, s)
    return np.r_[0, fps / fps[-1]], np.r_[0, tps / tps[-1]]


def auc_trapezoids(fpr, tpr):
    return float(np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2))


def auc_pairs(y, s):
    y, s = np.asarray(y), np.asarray(s)
    differences = s[y == 1][:, None] - s[y == 0][None, :]                  # all the (positive, negative) pairs
    return float(((differences > 0).sum() + 0.5 * (differences == 0).sum()) / differences.size)

In [ ]:
def average_precision(y, s):
    tps, fps = _cumulate(y, s)
    precision, recall = tps / (tps + fps), tps / tps[-1]
    return float(np.sum(np.diff(np.r_[0, recall]) * precision))


def best_f1_threshold(y, s):
    y, s = np.asarray(y), np.asarray(s)
    candidates = np.unique(s)
    f1 = [metrics(y, (s >= t).astype(int))["f1"] for t in candidates]
    k = int(np.argmax(f1))
    return float(candidates[k]), float(f1[k])

In [ ]:
fpr, tpr = roc_curve_(y_te, scores)
print("ROC points:", len(fpr), "| AUC (trapezoids):", round(auc_trapezoids(fpr, tpr), 4), "| AUC (pairs):", round(auc_pairs(y_te, scores), 4), "| scikit-learn:", round(roc_auc_score(y_te, scores), 4))
print("average precision:", round(average_precision(y_te, scores), 4), "| best threshold for F1:", best_f1_threshold(y_te, scores))

## Check

In [ ]:
with verify():
    pred = (scores >= 0.3).astype(int)
    check(confusion_matrix_(y_te, pred) == tuple(confusion_matrix(y_te, pred).ravel()), "correct confusion matrix", "tn, fp, fn, tp: mind the order")
    m = metrics(y_te, pred)
    check(close([m["precision"], m["recall"], m["f1"]], [precision_score(y_te, pred), recall_score(y_te, pred), f1_score(y_te, pred)]), "correct precision, recall and F1", "precision = TP/(TP+FP), recall = TP/(TP+FN), F1 = harmonic mean")
    check(m["accuracy"] == (pred == y_te).mean() and metrics(y_te, np.zeros_like(y_te))["precision"] == 0.0, "correct accuracy and no division by zero", "if you flag nothing the precision is 0 by convention")
    fpr, tpr = roc_curve_(y_te, scores)
    fpr_sk, tpr_sk, _ = roc_curve(y_te, scores, drop_intermediate=False)
    check(len(fpr) == len(np.unique(scores)) + 1, "one ROC point for every distinct value of the score (plus the origin)", "with ties you can't have a point for every example")
    check(close(fpr, fpr_sk) and close(tpr, tpr_sk), "the ROC curve matches scikit-learn's", "accumulate tp and fp only on the last indices of every group of equal scores")
    auc_sk = roc_auc_score(y_te, scores)
    check(close(auc_trapezoids(fpr, tpr), auc_sk) and close(auc_pairs(y_te, scores), auc_sk), "the AUC matches roc_auc_score in both ways", "pairs: (pos > neg) + 0.5 (pos == neg), divided by the number of pairs")
    check(close(average_precision(y_te, scores), average_precision_score(y_te, scores)), "correct average precision", "sum of (R_k - R_{k-1}) * P_k over the distinct values")
    s_best, f1_best = best_f1_threshold(y_te, scores)
    f1_brute_force = max(f1_score(y_te, (scores >= t).astype(int)) for t in np.unique(scores))
    check(close(f1_best, f1_brute_force) and close(f1_score(y_te, (scores >= s_best).astype(int)), f1_best), "the threshold found really maximizes F1", "try every distinct value of the score as a threshold")
done()

## What you learned

- The ROC curve is a list of cumulative counts: you sort the scores from the highest, and every extra example adds a true or a false positive. The **AUC is a probability**:
  pick a positive and a negative at random, how often does the positive have the higher score (ties count 1/2)? The two definitions (area and pairs) coincide.
- **Ties** must be grouped: if you move from one threshold to the next one example at a time inside a group of equal scores, you draw a diagonal that corresponds to
  no real threshold and the AUC changes. It is a bug that gives no errors.
- `average_precision` is the area of the PR curve **as a step sum** (not with trapezoids: interpolating linearly between PR points is optimistic).
- The threshold that maximizes F1 on the test set is an optimistic estimate: here you search for it only as an exercise, the real threshold must be chosen on the training set (lesson 4.3).

## Common pitfall

Computing the ROC by sorting the scores and taking a point for every **example** instead of every **distinct value**: with rounded or discrete scores (trees!)
the curve comes out different from scikit-learn's and the AUC slightly wrong, with no error.

## Interview question

<details><summary>How do you interpret an AUC of 0.78?</summary>

If I pick a defaulting customer and a non-defaulting one at random, in 78% of the cases the model gives the higher score to the defaulter. It says nothing about the
calibration of the probabilities nor about which threshold to use: it measures only the ordering. That is why with very imbalanced classes I pair it with precision and recall
at the chosen threshold or with the average precision.

</details>